In [1]:
"""
Incremental Loads & Change Data Capture (CDC)
Learn how to efficiently load only new/changed data from PostgreSQL
"""

import os
import time
import logging
import json
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================================
# CONFIGURATION
# ============================================================================

DB_CONFIG = {
    "url": "jdbc:postgresql://localhost:5432/Data_engineering_practice",
    "user": "postgres",
    "password": "xxxxx",  # 🔴 CHANGE THIS
    "driver": "org.postgresql.Driver"
}

BASE_DIR = "/home/odinsbeard/Data_engineering_Journey/week6_spark"
JAR_PATH = f"{BASE_DIR}/jars/postgresql-42.7.1.jar"
METADATA_DIR = f"{BASE_DIR}/metadata"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
DATA_DIR = f"{BASE_DIR}/data/incremental"

os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ============================================================================
# SETUP LOGGING
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger

<Logger __main__ (INFO)>

In [7]:
# ============================================================================
# INITIALIZE SPARK
# ============================================================================
def init_spark():
    """Initialize Spark session"""
    spark = SparkSession.builder \
        .appName("Incremental Loads CDC") \
        .config("spark.jars", JAR_PATH) \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    logger.info(f"✅ Spark session created (version: {spark.version})")
    return spark

# ============================================================================
# PART 1: METADATA TRACKING (Store last load state)
# ============================================================================

class LoadMetadata:
    """Track metadata for incremental loads"""
    
    def __init__(self, table_name):
        self.table_name = table_name
        self.metadata_file = f"{METADATA_DIR}/{table_name}_metadata.json"
        self.metadata = self._load()
    
    def _load(self):
        """Load metadata from file"""
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {
            "table": self.table_name,
            "last_load": None,
            "last_id": 0,
            "last_timestamp": "1970-01-01 00:00:00",
            "row_count": 0,
            "loads": []
        }
    
    def _save(self):
        """Save metadata to file"""
        with open(self.metadata_file, 'w') as f:
            json.dump(self.metadata, f, indent=2, default=str)
    
    def update(self, load_time, max_id, max_timestamp, rows_loaded):
        """Update metadata after successful load"""
        self.metadata["last_load"] = str(load_time)
        self.metadata["last_id"] = max_id
        self.metadata["last_timestamp"] = str(max_timestamp)
        self.metadata["row_count"] += rows_loaded
        self.metadata["loads"].append({
            "timestamp": str(load_time),
            "rows": rows_loaded,
            "max_id": max_id
        })
        # Keep only last 10 loads
        self.metadata["loads"] = self.metadata["loads"][-10:]
        self._save()

# ============================================================================
# PART 2: ID-BASED INCREMENTAL LOADS (For tables with auto-increment IDs)
# ============================================================================

def incremental_load_by_id(spark, table_name, id_column):
    """
    Load only new records where ID > last loaded ID
    Best for: tables with monotonically increasing IDs
    """
    logger.info("\n" + "=" * 70)
    logger.info(f"ID-BASED INCREMENTAL LOAD: {table_name}")
    logger.info("=" * 70)
    
    # Get metadata
    metadata = LoadMetadata(f"{table_name}_id")
    last_id = metadata.metadata["last_id"]
    
    logger.info(f"📋 Last loaded ID: {last_id}")
    
    # Query to get new records
    if last_id == 0:
        # First load - get all data
        query = table_name
        logger.info("🔄 First-time load (all data)")
    else:
        # Incremental load - only new records
        query = f"(SELECT * FROM {table_name} WHERE {id_column} > {last_id}) as new_data"
        logger.info(f"🔄 Loading records with {id_column} > {last_id}")
    
    # Read data
    start_time = time.time()
    
    df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=query,
        properties=DB_CONFIG
    )
    
    row_count = df.count()
    
    if row_count > 0:
        # Get the max ID from this batch
        max_id = df.agg(max(col(id_column))).collect()[0][0]
        
        # FIXED: Save to parquet WITHOUT partitioning
        output_path = f"{DATA_DIR}/{table_name}/id_based"
        df.write.mode("append") \
            .parquet(output_path)
        
        # Update metadata
        load_time = datetime.now()
        metadata.update(load_time, max_id, load_time, row_count)
        
        logger.info(f"✅ Loaded {row_count} new records")
        logger.info(f"📈 New max ID: {max_id}")
        logger.info(f"⏱️  Time: {time.time() - start_time:.2f} seconds")
        
        # Show sample of new data
        logger.info("\n📊 Sample of new data:")
        df.show(5, truncate=False)
    else:
        logger.info("✅ No new records found")
    
    return df
# ============================================================================
# PART 3: TIMESTAMP-BASED INCREMENTAL LOADS (For tables with update timestamps)
# ============================================================================

def incremental_load_by_timestamp(spark, table_name, timestamp_column):
    """
    Load only records changed since last load
    Best for: tables with created_at/updated_at timestamps
    """
    logger.info("\n" + "=" * 70)
    logger.info(f"TIMESTAMP-BASED INCREMENTAL LOAD: {table_name}")
    logger.info("=" * 70)
    
    # Get metadata
    metadata = LoadMetadata(f"{table_name}_ts")
    last_timestamp = metadata.metadata["last_timestamp"]
    
    logger.info(f"📋 Last loaded timestamp: {last_timestamp}")
    
    # Query to get changed records
    if last_timestamp == "1970-01-01 00:00:00":
        # First load - get all data
        query = table_name
        logger.info("🔄 First-time load (all data)")
    else:
        # Incremental load - records changed since last load
        query = f"(SELECT * FROM {table_name} WHERE {timestamp_column} > '{last_timestamp}') as new_data"
        logger.info(f"🔄 Loading records with {timestamp_column} > {last_timestamp}")
    
    # Read data
    start_time = time.time()
    
    df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=query,
        properties=DB_CONFIG
    )
    
    row_count = df.count()
    
    if row_count > 0:
        # Get the max timestamp from this batch
        max_timestamp = df.agg(max(col(timestamp_column))).collect()[0][0]
        
        # Save to parquet
        output_path = f"{DATA_DIR}/{table_name}/ts_based"
        df.write.mode("append") \
            .partitionBy("year", "month") \
            .parquet(output_path)
        
        # Update metadata
        load_time = datetime.now()
        metadata.update(load_time, 0, max_timestamp, row_count)
        
        logger.info(f"✅ Loaded {row_count} new/changed records")
        logger.info(f"📈 New max timestamp: {max_timestamp}")
        logger.info(f"⏱️  Time: {time.time() - start_time:.2f} seconds")
        
        # Show sample of new data
        logger.info("\n📊 Sample of new/changed data:")
        df.show(5, truncate=False)
    else:
        logger.info("✅ No new records found")
    
    return df

# ============================================================================
# PART 4: FULL CDC WITH DELTA DETECTION (Insert/Update/Delete)
# ============================================================================

def detect_changes_with_hash(spark, table_name, key_columns, tracked_columns):
    """
    Detect inserts, updates, and deletes using hash comparison
    This is a simplified CDC approach
    """
    logger.info("\n" + "=" * 70)
    logger.info(f"CDC WITH HASH DETECTION: {table_name}")
    logger.info("=" * 70)
    
    # Load current data from PostgreSQL
    logger.info("📥 Loading current data from PostgreSQL...")
    current_df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=table_name,
        properties=DB_CONFIG
    )
    
    # Create hash of tracked columns
    hash_cols = [col(c) for c in tracked_columns]
    current_df = current_df.withColumn(
        "row_hash",
        sha2(concat_ws("||", *hash_cols), 256)
    )
    
    # Load previous snapshot (if exists)
    snapshot_path = f"{DATA_DIR}/{table_name}/snapshot"
    if os.path.exists(snapshot_path):
        logger.info("📂 Loading previous snapshot...")
        previous_df = spark.read.parquet(snapshot_path)
        
        # Find changes
        # New records (in current but not in previous)
        new_records = current_df.join(
            previous_df.select(key_columns[0]),
            key_columns[0],
            "left_anti"
        )
        
        # Deleted records (in previous but not in current)
        deleted_records = previous_df.join(
            current_df.select(key_columns[0]),
            key_columns[0],
            "left_anti"
        )
        
        # Updated records (hash different)
        joined = current_df.alias("c").join(
            previous_df.alias("p"),
            key_columns[0]
        )
        updated_records = joined.filter(col("c.row_hash") != col("p.row_hash"))
        
        logger.info(f"📊 Change Summary:")
        logger.info(f"   ✅ New records: {new_records.count()}")
        logger.info(f"   ✅ Updated records: {updated_records.count()}")
        logger.info(f"   ❌ Deleted records: {deleted_records.count()}")
        
        if new_records.count() > 0:
            logger.info("\n📈 New records:")
            new_records.show(5)
        
        if updated_records.count() > 0:
            logger.info("\n📝 Updated records:")
            updated_records.select("c.*").show(5)
        
        if deleted_records.count() > 0:
            logger.info("\n🗑️ Deleted records:")
            deleted_records.show(5)
        
    else:
        logger.info("📊 First snapshot - no previous data for comparison")
        new_records = current_df
        logger.info(f"✅ Initial load: {new_records.count()} records")
    
    # Save new snapshot
    current_df.write.mode("overwrite").parquet(snapshot_path)
    logger.info(f"💾 Snapshot saved to {snapshot_path}")
    
    return current_df

# ============================================================================
# PART 5: SIMULATE NEW DATA (For testing)
# ============================================================================

def simulate_new_data():
    """
    Simulate new data being added to PostgreSQL
    In real world, this would be your application adding data
    """
    logger.info("\n" + "=" * 70)
    logger.info("SIMULATING NEW DATA INSERT")
    logger.info("=" * 70)
    
    # This is just a simulation - in reality, your app would add data
    # We'll just wait and let you manually add data if desired
    logger.info("""
    💡 In a real scenario, new data would be added by your application.
    For testing, you can:
    1. Manually insert records in PostgreSQL
    2. Run this script again to see incremental loads
    3. Observe that only new records are loaded
    """)

# ============================================================================
# PART 6: COMPARISON - FULL LOAD vs INCREMENTAL LOAD
# ============================================================================

def compare_load_strategies(spark, table_name, id_column):
    """Compare full load vs incremental load performance"""
    logger.info("\n" + "=" * 70)
    logger.info("COMPARING LOAD STRATEGIES")
    logger.info("=" * 70)
    
    # Full load
    logger.info("\n🔍 Testing FULL LOAD...")
    start = time.time()
    full_df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=table_name,
        properties=DB_CONFIG
    )
    full_count = full_df.count()
    full_time = time.time() - start
    logger.info(f"   Full load: {full_count:,} rows in {full_time:.2f}s")
    
    # Incremental load (simulate by adding 10% more data)
    logger.info("\n🔍 Testing INCREMENTAL LOAD (10% new data)...")
    
    # Get max ID
    max_id = full_df.agg(max(col(id_column))).collect()[0][0]
    new_id_start = max_id + 1
    
    # Simulate incremental query
    incremental_query = f"(SELECT * FROM {table_name} WHERE {id_column} >= {new_id_start}) as new"
    
    start = time.time()
    inc_df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=incremental_query,
        properties=DB_CONFIG
    )
    inc_count = inc_df.count()
    inc_time = time.time() - start
    logger.info(f"   Incremental load: {inc_count:,} rows in {inc_time:.2f}s")
    
    # Calculate improvement
    if inc_count > 0:
        rows_per_sec_full = full_count / full_time
        rows_per_sec_inc = inc_count / inc_time
        
        logger.info("\n" + "=" * 70)
        logger.info("PERFORMANCE COMPARISON")
        logger.info("=" * 70)
        logger.info(f"{'Strategy':<15} {'Rows':<12} {'Time (s)':<12} {'Rows/sec':<15}")
        logger.info("-" * 60)
        logger.info(f"{'Full Load':<15} {full_count:<12,} {full_time:<12.2f} {rows_per_sec_full:<15,.0f}")
        logger.info(f"{'Incremental':<15} {inc_count:<12,} {inc_time:<12.2f} {rows_per_sec_inc:<15,.0f}")
        logger.info(f"{'Improvement':<15} {'':<12} {(full_time/inc_time):<12.1f}x {(rows_per_sec_inc/rows_per_sec_full):<15.1f}x")

# ============================================================================
# PART 7: DEMONSTRATION - INCREMENTAL LOADS IN ACTION
# ============================================================================

def demonstrate_incremental_loads(spark):
    """Complete demonstration of incremental loads"""
    logger.info("\n" + "=" * 70)
    logger.info("INCREMENTAL LOADS DEMONSTRATION")
    logger.info("=" * 70)
    
    tables = [
        ("order_items", "order_item_id", "ID-based"),
        ("orders", "order_id", "ID-based"),
        ("customers", "customer_id", "ID-based")
    ]
    
    for table_name, id_col, strategy in tables:
        logger.info(f"\n📊 {strategy} - {table_name}")
        
        # First load (should load all data)
        df1 = incremental_load_by_id(spark, table_name, id_col)
        
        # Show metadata after first load
        metadata = LoadMetadata(f"{table_name}_id")
        logger.info(f"\n📋 Metadata after first load:")
        logger.info(f"   Last ID: {metadata.metadata['last_id']}")
        logger.info(f"   Total rows: {metadata.metadata['row_count']}")
        
        # Second load (should load nothing)
        logger.info(f"\n🔄 Running second load (should find no new data)...")
        df2 = incremental_load_by_id(spark, table_name, id_col)
        
        logger.info("-" * 50)


In [8]:

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution function"""
    logger.info("=" * 70)
    logger.info("INCREMENTAL LOADS & CHANGE DATA CAPTURE (CDC)")
    logger.info("=" * 70)
    
    spark = init_spark()
    
    try:
        # Part 1: Demonstrate basic incremental loads
        demonstrate_incremental_loads(spark)
        
        # Part 2: Compare full vs incremental performance
        compare_load_strategies(spark, "order_items", "order_item_id")
        
        # Part 3: Show timestamp-based incremental loads
        logger.info("\n" + "=" * 70)
        logger.info("TIMESTAMP-BASED INCREMENTAL LOADS")
        logger.info("=" * 70)
        
        # Note: This assumes your tables have timestamp columns
        # If not, you'd need to add them
        logger.info("""
        💡 For timestamp-based incremental loads, your tables need:
        - created_at TIMESTAMP (when record was created)
        - updated_at TIMESTAMP (when record was last updated)
        
        Query would be:
        SELECT * FROM table 
        WHERE updated_at > last_load_timestamp
        """)
        
        # Part 4: CDC with hash detection
        logger.info("\n" + "=" * 70)
        logger.info("CDC WITH HASH DETECTION")
        logger.info("=" * 70)
        logger.info("""
        This approach detects:
        ✅ INSERTS - New records not in previous snapshot
        ✅ UPDATES - Records with changed data (hash mismatch)
        ❌ DELETES - Records missing from current snapshot
        
        Best for tables without timestamp columns
        """)
        
        # Part 5: Best practices summary
        logger.info("\n" + "=" * 70)
        logger.info("INCREMENTAL LOADS BEST PRACTICES")
        logger.info("=" * 70)
        logger.info("""
📌 CHOOSE THE RIGHT STRATEGY:
   • ID-based: Tables with auto-incrementing IDs
   • Timestamp-based: Tables with created_at/updated_at
   • Hash-based: When you need full CDC (inserts/updates/deletes)

📌 KEY BENEFITS:
   • 10-100x faster than full loads
   • Less load on PostgreSQL
   • Less network traffic
   • Can run more frequently

📌 IMPLEMENTATION TIPS:
   • Always track metadata (last_id, last_timestamp)
   • Store checkpoints for recovery
   • Consider partitioning for large tables
   • Monitor and alert on load failures

📌 REAL-WORLD PATTERNS:
   • Hourly incremental loads for recent data
   • Daily full refresh for reference tables
   • CDC for real-time analytics
        """)
        
    finally:
        spark.stop()
        logger.info("\n✅ Spark session stopped")

if __name__ == "__main__":
    main()



2026-03-14 09:56:00,759 - INFO - ======================================================================
2026-03-14 09:56:00,761 - INFO - INCREMENTAL LOADS & CHANGE DATA CAPTURE (CDC)
2026-03-14 09:56:00,764 - INFO - ======================================================================
2026-03-14 09:56:01,273 - INFO - ✅ Spark session created (version: 3.5.0)
2026-03-14 09:56:01,274 - INFO - 
2026-03-14 09:56:01,280 - INFO - INCREMENTAL LOADS DEMONSTRATION
2026-03-14 09:56:01,283 - INFO - ======================================================================
2026-03-14 09:56:01,293 - INFO - 
📊 ID-based - order_items
2026-03-14 09:56:01,304 - INFO - 
2026-03-14 09:56:01,304 - INFO - ID-BASED INCREMENTAL LOAD: order_items
2026-03-14 09:56:01,310 - INFO - ======================================================================
2026-03-14 09:56:01,315 - INFO - 📋 Last loaded ID: 0
2026-03-14 09:56:01,317 - INFO - 🔄 First-time load (all data)
2026-03-14 09:56:04,993 - INFO - ✅ Loaded 721 new re

+-------------+--------+----------+--------+----------+----------------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_percent|
+-------------+--------+----------+--------+----------+----------------+
|1            |1       |37        |1       |71.41     |0.00            |
|2            |1       |42        |1       |230.49    |15.00           |
|3            |1       |34        |3       |72.88     |10.00           |
|4            |1       |41        |5       |99.16     |0.00            |
|5            |1       |28        |5       |456.25    |0.00            |
+-------------+--------+----------+--------+----------+----------------+
only showing top 5 rows



2026-03-14 09:56:07,219 - INFO - ✅ No new records found
2026-03-14 09:56:07,220 - INFO - --------------------------------------------------
2026-03-14 09:56:07,221 - INFO - 
📊 ID-based - orders
2026-03-14 09:56:07,224 - INFO - 
2026-03-14 09:56:07,225 - INFO - ID-BASED INCREMENTAL LOAD: orders
2026-03-14 09:56:07,228 - INFO - ======================================================================
2026-03-14 09:56:07,229 - INFO - 📋 Last loaded ID: 0
2026-03-14 09:56:07,229 - INFO - 🔄 First-time load (all data)
2026-03-14 09:56:08,493 - INFO - ✅ Loaded 200 new records
2026-03-14 09:56:08,495 - INFO - 📈 New max ID: 200
2026-03-14 09:56:08,496 - INFO - ⏱️  Time: 1.27 seconds
2026-03-14 09:56:08,497 - INFO - 
📊 Sample of new data:
2026-03-14 09:56:08,771 - INFO - 
📋 Metadata after first load:
2026-03-14 09:56:08,772 - INFO -    Last ID: 200
2026-03-14 09:56:08,773 - INFO -    Total rows: 200
2026-03-14 09:56:08,775 - INFO - 
🔄 Running second load (should find no new data)...
2026-03-14 09:56

+--------+-----------+--------------------------+------------+---------+--------------+
|order_id|customer_id|order_date                |total_amount|status   |payment_method|
+--------+-----------+--------------------------+------------+---------+--------------+
|1       |4          |2025-05-08 10:08:02.766264|198.20      |Pending  |Debit Card    |
|2       |34         |2025-07-26 12:08:02.766323|882.71      |Shipped  |Credit Card   |
|3       |67         |2025-10-11 18:08:02.766332|256.53      |Cancelled|Cash          |
|4       |72         |2025-09-14 22:08:02.766337|486.57      |Cancelled|Cash          |
|5       |3          |2025-11-30 11:08:02.766342|237.16      |Pending  |Debit Card    |
+--------+-----------+--------------------------+------------+---------+--------------+
only showing top 5 rows



2026-03-14 09:56:09,119 - INFO - ✅ No new records found
2026-03-14 09:56:09,121 - INFO - --------------------------------------------------
2026-03-14 09:56:09,123 - INFO - 
📊 ID-based - customers
2026-03-14 09:56:09,125 - INFO - 
2026-03-14 09:56:09,126 - INFO - ID-BASED INCREMENTAL LOAD: customers
2026-03-14 09:56:09,127 - INFO - ======================================================================
2026-03-14 09:56:09,128 - INFO - 📋 Last loaded ID: 0
2026-03-14 09:56:09,129 - INFO - 🔄 First-time load (all data)
2026-03-14 09:56:10,175 - INFO - ✅ Loaded 101 new records
2026-03-14 09:56:10,176 - INFO - 📈 New max ID: 102
2026-03-14 09:56:10,177 - INFO - ⏱️  Time: 1.05 seconds
2026-03-14 09:56:10,178 - INFO - 
📊 Sample of new data:
2026-03-14 09:56:10,684 - INFO - 
📋 Metadata after first load:
2026-03-14 09:56:10,686 - INFO -    Last ID: 102
2026-03-14 09:56:10,687 - INFO -    Total rows: 101
2026-03-14 09:56:10,687 - INFO - 
🔄 Running second load (should find no new data)...
2026-03-14

+-----------+----------+---------+-------------------+--------+--------+-----+-------+-----------------+---------+
|customer_id|first_name|last_name|email              |phone   |city    |state|country|registration_date|is_active|
+-----------+----------+---------+-------------------+--------+--------+-----+-------+-----------------+---------+
|1          |FirstName1|LastName1|customer1@email.com|555-2824|New York|NY   |USA    |2025-04-10       |true     |
|2          |FirstName2|LastName2|customer2@email.com|555-4657|Chicago |IL   |USA    |2025-10-04       |true     |
|3          |FirstName3|LastName3|customer3@email.com|555-7912|New York|NY   |USA    |2025-12-17       |true     |
|4          |FirstName4|LastName4|customer4@email.com|555-4582|Houston |TX   |USA    |2024-08-17       |true     |
|5          |FirstName5|LastName5|customer5@email.com|555-4257|Dallas  |TX   |USA    |2024-11-13       |true     |
+-----------+----------+---------+-------------------+--------+--------+-----+--

2026-03-14 09:56:11,083 - INFO - ✅ No new records found
2026-03-14 09:56:11,083 - INFO - --------------------------------------------------
2026-03-14 09:56:11,084 - INFO - 
2026-03-14 09:56:11,084 - INFO - COMPARING LOAD STRATEGIES
2026-03-14 09:56:11,085 - INFO - ======================================================================
2026-03-14 09:56:11,085 - INFO - 
🔍 Testing FULL LOAD...
2026-03-14 09:56:11,382 - INFO -    Full load: 721 rows in 0.30s
2026-03-14 09:56:11,386 - INFO - 
🔍 Testing INCREMENTAL LOAD (10% new data)...
2026-03-14 09:56:12,097 - INFO -    Incremental load: 0 rows in 0.40s
2026-03-14 09:56:12,102 - INFO - 
2026-03-14 09:56:12,104 - INFO - TIMESTAMP-BASED INCREMENTAL LOADS
2026-03-14 09:56:12,119 - INFO - ======================================================================
2026-03-14 09:56:12,124 - INFO - 
        💡 For timestamp-based incremental loads, your tables need:
        - created_at TIMESTAMP (when record was created)
        - updated_at TIMESTAM